# KNN con k = 57 — versión para explicar

Este notebook muestra únicamente el modelo KNN con los parámetros que resultaron seleccionados en el análisis técnico: **57 vecinos** y **ponderación por distancia**. No busca el mejor `k`; esa búsqueda está documentada en el notebook 03.

La pregunta que responde es: *para un cliente nuevo, ¿cómo se usan sus 57 clientes históricos más parecidos para estimar su probabilidad de churn?*

## 1. Importar las herramientas

- `pandas` lee y organiza los datos.
- `scikit-learn` prepara variables, separa la validación y entrena KNN.
- Las gráficas permiten evaluar el modelo sin llenar el notebook de mensajes de texto.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RANDOM_STATE = 2026
K = 57

RAIZ = Path.cwd().resolve()
if RAIZ.name == 'notebooks':
    RAIZ = RAIZ.parent
RUTA_DATOS = RAIZ / 'data' / 'processed'

sns.set_theme(style='whitegrid', context='notebook')

## 2. Cargar la base preparada

Usamos la salida del notebook 01. Allí ya se limpiaron los nombres de columnas y se crearon edad y antigüedad contractual. `resultado` vale 1 si hubo churn y 0 si no lo hubo.

In [ ]:
train = pd.read_csv(RUTA_DATOS / 'train_preparado.csv')

# id identifica a la persona, pero no se usa para predecir.
# Las fechas originales ya están representadas por edad y antigüedad de contrato.
COLUMNAS_EXCLUIDAS = ['id', 'fecha_nacimiento', 'fecha_inicio_contrato', 'resultado']
CATEGORICAS = ['tipo_cliente', 'factura_online', 'plan_datos', 'edad_invalida', 'antiguedad_contrato_invalida']
NUMERICAS = [c for c in train.columns if c not in COLUMNAS_EXCLUIDAS + CATEGORICAS]
PREDICTORES = NUMERICAS + CATEGORICAS

X = train[PREDICTORES]
y = train['resultado']

display(pd.DataFrame({
    'grupo': ['Variables numéricas', 'Variables categóricas', 'Clientes para entrenar'],
    'detalle': [', '.join(NUMERICAS), ', '.join(CATEGORICAS), len(train)]
}))

## 3. Reservar clientes para validar

El modelo aprende con 80% de los clientes. El 20% restante se reserva para revisar si predice bien casos que no vio durante el aprendizaje. `stratify=y` conserva aproximadamente la misma tasa de churn en ambos grupos.

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

display(pd.DataFrame({
    'muestra': ['Entrenamiento', 'Validación'],
    'clientes': [len(X_train), len(X_val)],
    'tasa de churn': [y_train.mean(), y_val.mean()]
}).style.format({'tasa de churn': '{:.1%}'}))

## 4. Preparar las variables

KNN calcula distancias. Por eso no podemos comparar una facturación de cientos de miles con días de mora sin ajustar escalas: la facturación dominaría la distancia.

- Las numéricas se imputan por mediana y luego se estandarizan.
- Las categóricas se convierten en columnas 0/1.
- Todo ocurre dentro del `Pipeline`, evitando que la información de validación se filtre al entrenamiento.

In [ ]:
preparacion_numerica = Pipeline([
    ('imputar', SimpleImputer(strategy='median')),
    ('escalar', StandardScaler())
])

preparacion_categorica = Pipeline([
    ('imputar', SimpleImputer(strategy='most_frequent')),
    ('convertir_a_0_1', OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

preprocesador = ColumnTransformer([
    ('numericas', preparacion_numerica, NUMERICAS),
    ('categoricas', preparacion_categorica, CATEGORICAS),
])

## 5. Definir y entrenar KNN

`n_neighbors=57` indica que la predicción de cada cliente se basa en sus 57 clientes históricos más cercanos. `weights='distance'` da más importancia a quienes están más cerca; por eso no todos los 57 votos pesan igual.

In [ ]:
knn_57 = KNeighborsClassifier(
    n_neighbors=K,
    weights='distance'
)

modelo = Pipeline([
    ('preprocesar', preprocesador),
    ('knn', knn_57)
])

# fit aprende las escalas, la codificación y guarda los clientes de entrenamiento como referencia.
modelo.fit(X_train, y_train)

## 6. Evaluar con probabilidades, no con una etiqueta fija

`predict_proba` devuelve una probabilidad de churn para cada cliente de validación. AUC-ROC evalúa qué tan bien el modelo ordena clientes de menor a mayor riesgo; no depende de escoger un único punto de corte.

In [ ]:
probabilidad_churn = modelo.predict_proba(X_val)[:, 1]
auc_validacion = roc_auc_score(y_val, probabilidad_churn)
fpr, tpr, _ = roc_curve(y_val, probabilidad_churn)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(fpr, tpr, label=f'KNN, k=57 (AUC = {auc_validacion:.4f})')
ax.plot([0, 1], [0, 1], '--', color='gray', label='Azar')
ax.set(xlabel='Tasa de falsos positivos', ylabel='Tasa de verdaderos positivos', title='Curva ROC en validación')
ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

## 7. Mirar los 57 vecinos de un cliente concreto

Elegimos el primer cliente de validación solo como ejemplo. Primero transformamos sus variables a la misma escala del entrenamiento; después KNN calcula la distancia hacia todos los clientes de entrenamiento y conserva los 57 más cercanos.

La tabla muestra los diez vecinos más próximos. La probabilidad final usa los 57, con más peso para las distancias menores.

In [ ]:
CLIENTE_EJEMPLO = 0

# Obtenemos la representación numérica que KNN realmente usa para medir distancia.
X_train_preparado = modelo.named_steps['preprocesar'].transform(X_train)
X_val_preparado = modelo.named_steps['preprocesar'].transform(X_val)

# kneighbors devuelve las distancias y las posiciones de los 57 vecinos más cercanos.
distancias, posiciones = modelo.named_steps['knn'].kneighbors(
    X_val_preparado[[CLIENTE_EJEMPLO]],
    n_neighbors=K
)

vecinos = X_train.iloc[posiciones[0]].copy()
vecinos['resultado_historico'] = y_train.iloc[posiciones[0]].to_numpy()
vecinos['distancia'] = distancias[0]

resumen_vecindario = pd.DataFrame({
    'cliente de validación': [X_val.index[CLIENTE_EJEMPLO]],
    'probabilidad estimada de churn': [probabilidad_churn[CLIENTE_EJEMPLO]],
    'vecinos con churn entre 57': [vecinos.resultado_historico.sum()],
    'vecinos sin churn entre 57': [(vecinos.resultado_historico == 0).sum()]
})
display(resumen_vecindario.style.format({'probabilidad estimada de churn': '{:.1%}'}))
display(vecinos[['resultado_historico', 'distancia', 'facturacion', 'mora', 'minutos']].head(10).style.format({'distancia': '{:.3f}', 'facturacion': '{:,.0f}'}))

## Idea final para explicar

Con `k=57`, KNN no crea 57 segmentos. Para **cada** cliente que quiere clasificar, busca 57 referencias cercanas y calcula una probabilidad a partir de sus resultados históricos y sus distancias. Otro cliente podría tener un conjunto de 57 vecinos completamente diferente.